# Advanced RNN Architectures

In the previous notebooks, we used simple, single-layer RNNs/LSTMs. To improve performance on complex tasks, we can use more advanced techniques.

## 1. Bidirectional RNNs
Standard RNNs process data only in one direction (forward). However, for many tasks (like translation or speech), the future context is just as important as the past. 

**Bidirectional RNNs** consist of two independent RNNs together: one processes the input sequence in the forward direction, and the other in the backward direction. Their outputs are usually concatenated at each time step.

## 2. Stacked (Deep) RNNs
Just like with Dense networks, we can stack multiple RNN layers on top of each other to learn more complex features. 

> **Crucial**: To stack RNN layers in Keras, all layers except the last one must return the full sequence of outputs, not just the final state. We do this by setting `return_sequences=True`.

## 3. Regularization: Dropout
RNNs are prone to overfitting. We can apply:
- **Dropout**: Applied to the inputs of the layer.
- **Recurrent Dropout**: Applied to the recurrent state (between time steps).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout

tf.random.set_seed(42)

## Data Loading (IMDB)

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
X_train = pad_sequences(X_train, maxlen=MAX_LEN)
X_test = pad_sequences(X_test, maxlen=MAX_LEN)

## Building an Advanced Model
We will build a model that uses:
1. **Embedding Layer**
2. **Bidirectional LSTM Layer** (returning sequences)
3. **Second LSTM Layer** (stacked on top)
4. **Dense Output**

In [ ]:
embedding_dim = 64

model = Sequential()
model.add(Embedding(input_dim=VOCAB_SIZE, output_dim=embedding_dim, input_length=MAX_LEN))

# Layer 1: Bidirectional LSTM, returns sequences for the next layer
model.add(Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)))

# Layer 2: Standard LSTM to consume the sequence and produce a single vector
model.add(LSTM(32, dropout=0.2))

# Output Layer
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## Training
*Note: Bidirectional and Stacked RNNs are much more computationally expensive than simple ones. Training will take longer.*

In [ ]:
history = model.fit(X_train, y_train, epochs=3, batch_size=64, validation_split=0.2, verbose=1)

## Conclusion
By stacking layers and processing sequences bidirectionally, we enable the model to learn much richer representations of the text. This is the foundation for modern NLP architectures (before Transformers took over).